In [0]:
%pip install openpyxl

In [0]:
import pandas as pd
import openpyxl

wb = openpyxl.load_workbook("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", read_only=True)
sheet_names = wb.sheetnames
display(sheet_names)

for sheet_name in sheet_names:
    df_sheet = pd.read_excel("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_raw.xlsx", sheet_name=sheet_name)
    df_sheet['sheet_name'] = sheet_name
    df_sheet['row_number'] = range(1, len(df_sheet) + 1)
    spark_df = spark.createDataFrame(df_sheet)
    spark_df.write.format("delta").mode("append").saveAsTable("sandbox.z_yeswook_kim.jira_1410_raw")

In [0]:
%sql
select 
    sheet_name, count(1)
from sandbox.z_yeswook_kim.jira_1410_raw
group by 
    sheet_name

In [0]:
%sql
select *
from sandbox.z_yeswook_kim.jira_1410_raw

In [0]:
%sql
select 
    sheet_name,
    row_number,
    CSN,
    MAC_ADDRESS,
    concat_ws(':',
        substring(MAC_ADDRESS, 1, 2),
        substring(MAC_ADDRESS, 3, 2),
        substring(MAC_ADDRESS, 5, 2),
        substring(MAC_ADDRESS, 7, 2),
        substring(MAC_ADDRESS, 9, 2),
        substring(MAC_ADDRESS, 11, 2)
    ) as mac_addr_origin,
    kic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
from sandbox.z_yeswook_kim.jira_1410_raw
limit 50

In [0]:
%sql
create or replace table sandbox.z_yeswook_kim.jira_1410_l1
select 
    sheet_name,
    row_number,
    CSN,
    MAC_ADDRESS,
    concat_ws(':',
        substring(MAC_ADDRESS, 1, 2),
        substring(MAC_ADDRESS, 3, 2),
        substring(MAC_ADDRESS, 5, 2),
        substring(MAC_ADDRESS, 7, 2),
        substring(MAC_ADDRESS, 9, 2),
        substring(MAC_ADDRESS, 11, 2)
    ) as mac_addr_origin,
    kic_data_ods.tlamp.hash_mac(mac_addr_origin) as mac_addr
from sandbox.z_yeswook_kim.jira_1410_raw
-- group by 
    -- sheet_name

In [0]:
%sql
select *
from sandbox.z_yeswook_kim.jira_1410_l1

In [0]:
%sql
    -- SELECT *
    -- FROM sandbox.z_yeswook_kim.jira_1410_l1

In [0]:
%sql
-- 전체 쿼리: 모든 normal_log 버전 포함
create or replace table sandbox.z_yeswook_kim.jira_1410_l2
WITH base AS (
    SELECT *
    FROM sandbox.z_yeswook_kim.jira_1410_l1
),

-- activation_date: 기기별 최초 crt_date, 마지막 last_chg_date
act AS (
    SELECT 
        mac_addr,
        MIN(crt_date) AS crt_date,
        MAX(last_chg_date) AS last_chg_date
    FROM kic_data_ods.tlamp.activation_date
    WHERE mac_addr IN (SELECT mac_addr FROM base)
    GROUP BY mac_addr
),

-- normal_log: 6개 버전 UNION ALL (tvpowerd + NL_POWER_STATE 한정)
nl AS (
    SELECT 
        mac_addr,
        MAX(log_create_time) AS log_create_time,
        MAX(accum_run_time) AS accum_run_time
    FROM (
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos60
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos22
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos23
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos24
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos25
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
        UNION ALL
        SELECT mac_addr, log_create_time, accum_run_time
        FROM kic_data_ods.tlamp.normal_log_webos26
        WHERE context_name = 'tvpowerd' AND message_id = 'NL_POWER_STATE'
          AND mac_addr IN (SELECT mac_addr FROM base)
    )
    GROUP BY mac_addr
)

SELECT 
    b.*,
    act.crt_date,
    act.last_chg_date,
    nl.log_create_time,
    nl.accum_run_time
FROM base b
LEFT JOIN act ON b.mac_addr = act.mac_addr
LEFT JOIN nl ON b.mac_addr = nl.mac_addr


In [0]:
%sql
select 
    sheet_name, count(1), count(crt_date), count(last_chg_date), count(log_create_time), count(last_chg_date)
from sandbox.z_yeswook_kim.jira_1410_l2
group by sheet_name

In [0]:
from pyspark.sql import SparkSession

df = spark.table("sandbox.z_yeswook_kim.jira_1410_l2")
sheet_names = [row.sheet_name for row in df.select("sheet_name").distinct().collect()]

for sheet_name in sheet_names:
    df_sheet = df.filter(df.sheet_name == sheet_name)
    output_path = f"/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira_1410_l2_csv/{sheet_name}.csv"
    df_sheet.coalesce(1).write.option("header", "true").mode("overwrite").csv(output_path)

In [0]:
%sql
select *
from sandbox.z_yeswook_kim.jira_1410_l2
where 1=1
    and log_create_time is not null